# PDB Fetch Tools

This notebook demonstrates the two PDB fetch functions provided by proto_tools. `run_pdb_fetch_entry` retrieves entry-level metadata from the RCSB REST API, returning the structure title, experimental method, and resolution for a given accession. `run_pdb_fetch_fasta` fetches the FASTA sequences for all polymer chains in a structure and automatically classifies each chain as protein or nucleotide based on its composition. Together these functions support common protein design workflows where you need to retrieve a structural template and its associated sequences before running inverse folding, mutagenesis, or structure prediction.

In [1]:
from proto_tools.utils.notebook_docs import display_overview, display_api_reference, display_docs_section, display_doc_link, display_available_tools
display_doc_link("pdb")
display_overview("pdb")
display_docs_section("pdb", "Background")

# PDB

> Two tools for retrieving data from the RCSB Protein Data Bank:

## Background

**What does this tool measure/predict?**
These tools retrieve information from the [RCSB Protein Data Bank](https://www.rcsb.org/), the global archive for experimentally determined 3D structures of biological macromolecules. PDB entries contain structures solved by [X-ray crystallography](https://en.wikipedia.org/wiki/X-ray_crystallography), [cryo-EM](https://en.wikipedia.org/wiki/Cryogenic_electron_microscopy), [NMR spectroscopy](https://en.wikipedia.org/wiki/Nuclear_magnetic_resonance_spectroscopy_of_proteins), and other methods.

**Why is this important?**

* Structure quality assessment: resolution and experimental method indicate reliability of atomic coordinates
* Protein design: extract reference sequences from specific chains of experimental structures
* Structural analysis: identify which chains are protein vs nucleic acid in multi-component complexes
* Benchmarking: retrieve experimental structures to compare against computational predictions

**Scientific foundation:**
The PDB was established in 1971 and now contains over 220,000 structures. Key metadata:

* **Resolution** (for X-ray/cryo-EM): lower values indicate higher-quality atomic models (1.0-2.0 A is considered high resolution)
* **Experimental method**: X-ray crystallography (most entries), cryo-EM (growing), NMR (small proteins), neutron diffraction (rare)
* **Chain classification**: protein chains contain amino acids; nucleotide chains contain DNA/RNA bases

## Available tools

In [2]:
display_available_tools("pdb")

- **`run_pdb_fetch_entry()`** — Fetch structure metadata (title, method, resolution) from RCSB PDB
- **`run_pdb_fetch_fasta()`** — Fetch chain sequences from RCSB PDB with protein/nucleotide classification

### `run_pdb_fetch_entry`

`run_pdb_fetch_entry` queries the RCSB REST API to retrieve catalog-level information about a PDB structure: the structure title, experimental method (e.g., X-RAY DIFFRACTION, SOLUTION NMR), and resolution in angstroms. This is useful as a quick lookup step before downloading full structure files, allowing you to confirm the experimental quality and method of a candidate template. We use PDB entry 1LBG, the lac repressor bound to a symmetric operator DNA, as the running example.

In [3]:
from proto_tools import (
    PdbFetchConfig,
    PdbFetchEntryInput,
    run_pdb_fetch_entry,
)

In [4]:
# Display docs
display_api_reference("pdb", "input", "run_pdb_fetch_entry")

# Example input
inputs = PdbFetchEntryInput(pdb_id="1LBG")

**Input** — `PdbFetchEntryInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `pdb_id` | `string` | required | PDB accession (e.g. '1LBG'). |

In [5]:
# Display config docs
display_api_reference("pdb", "config", "run_pdb_fetch_entry")

# Config is shared across both PDB fetch functions | see docs above for all fields
config = PdbFetchConfig()

**Config** — `PdbFetchConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `request_timeout_seconds` | `integer` | `15` | HTTP timeout per request. |
| `http_retries` | `integer` | `3` | Maximum HTTP retries. |
| `backoff_seconds` | `number` | `1.0` | Seconds to wait between retries (doubles after each attempt). |
| `user_agent` | `string` | `proto-tools/pdb-fetch-v1` | Identifier string sent to database APIs with each request. |
| `seed` | `integer` |  | Random seed for reproducible results. When set, same seed + same input produces identical output. When None, a random seed is auto-assigned internally via the `resolved_seed` property. Some GPU tools produce approximately (not bit-exact) identical results because CUDA operations reduce floating-point values in non-deterministic thread order. Discrete outputs (sequences, structures) are exact; floating-point outputs (scores, coordinates) may differ at the bit level (\~1e-7). |
| `verbose` | `boolean` | `False` | Whether to print status messages. |
| `device` | `string` | `cpu` | Device to run the tool on. |
| `timeout` | `integer` | `600` | Maximum execution time in seconds. |

In [6]:
# Run the entry fetch
entry_output = run_pdb_fetch_entry(inputs, config)

In [7]:
# Display output docs
display_api_reference("pdb", "output", "run_pdb_fetch_entry")

# Inspect the structure metadata
print(f"Title: {entry_output.title}")
print(f"Method: {entry_output.method}")
print(f"Resolution: {entry_output.resolution} A")
print(f"Source: {entry_output.source_url}")

**Output** — `PdbFetchEntryOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `title` | `string` |  | Structure title. |
| `method` | `string` |  | Experimental method. |
| `resolution` | `number` |  | Resolution in angstroms. |
| `source_url` | `string` |  | URL used for the request. |

Title: LACTOSE OPERON REPRESSOR BOUND TO 21-BASE PAIR SYMMETRIC OPERATOR DNA, ALPHA CARBONS ONLY
Method: X-RAY DIFFRACTION
Resolution: 4.8 A
Source: https://data.rcsb.org/rest/v1/core/entry/1LBG


### `run_pdb_fetch_fasta`

`run_pdb_fetch_fasta` retrieves the polymer chain sequences from RCSB PDB in FASTA format and automatically classifies each chain as protein or nucleotide based on its amino acid composition. A PDB structure can contain multiple chains of different types, so this function is useful when you need to extract specific chains for downstream analysis, such as feeding a protein chain into a structure prediction model or aligning it against homologous sequences.

In [8]:
from proto_tools import (
    PdbFetchFastaInput,
    run_pdb_fetch_fasta,
)

In [9]:
# Display docs
display_api_reference("pdb", "input", "run_pdb_fetch_fasta")

# Fetch chain sequences for the lac repressor structure
fasta_inputs = PdbFetchFastaInput(pdb_id="1LBG")

**Input** — `PdbFetchFastaInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `pdb_id` | `string` | required | PDB accession (e.g. '1LBG'). |

In [10]:
# Display config docs
display_api_reference("pdb", "config", "run_pdb_fetch_fasta")

# Config is shared across both PDB fetch functions | see docs above for all fields
fasta_config = PdbFetchConfig()

**Config** — `PdbFetchConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `request_timeout_seconds` | `integer` | `15` | HTTP timeout per request. |
| `http_retries` | `integer` | `3` | Maximum HTTP retries. |
| `backoff_seconds` | `number` | `1.0` | Seconds to wait between retries (doubles after each attempt). |
| `user_agent` | `string` | `proto-tools/pdb-fetch-v1` | Identifier string sent to database APIs with each request. |
| `seed` | `integer` |  | Random seed for reproducible results. When set, same seed + same input produces identical output. When None, a random seed is auto-assigned internally via the `resolved_seed` property. Some GPU tools produce approximately (not bit-exact) identical results because CUDA operations reduce floating-point values in non-deterministic thread order. Discrete outputs (sequences, structures) are exact; floating-point outputs (scores, coordinates) may differ at the bit level (\~1e-7). |
| `verbose` | `boolean` | `False` | Whether to print status messages. |
| `device` | `string` | `cpu` | Device to run the tool on. |
| `timeout` | `integer` | `600` | Maximum execution time in seconds. |

In [11]:
# Run the FASTA fetch
fasta_output = run_pdb_fetch_fasta(fasta_inputs, fasta_config)

In [12]:
# Display output docs
display_api_reference("pdb", "output", "run_pdb_fetch_fasta")

# Inspect the chain sequences and their classifications
print(f"Chains: {len(fasta_output.chains)}")
print()

for chain in fasta_output.chains:
    chain_type = "protein" if chain.is_protein else "nucleotide"
    preview = chain.sequence[:50] + ("..." if len(chain.sequence) > 50 else "")
    print(f"  Chain {chain.chain_id}: {chain_type}, length={len(chain.sequence)}")
    print(f"    {preview}")

**Output** — `PdbFetchFastaOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `chains` | `List[PdbChain]` |  | Parsed chain sequences with protein/nucleotide classification. |
| `chain_id` | `string` |  | Chain identifier extracted from header. |
| `header` | `string` | required | Full FASTA header line. |
| `sequence` | `string` | required | Chain sequence string. |
| `is_protein` | `boolean` | required | True if chain is protein, False if nucleic acid. |
| `source_url` | `string` |  | URL used for the request. |

Chains: 2

  Chain 2: protein, length=360
    MKPVTLYDVAEYAGVSYQTVSRVVNQASHVSAKTREKVEAAMAELNYIPN...
  Chain 1: nucleotide, length=21
    GAATTGTGAGCGCTCACAATT


## Export Results

Fetched results can be saved to JSON format for downstream use in other tools or analysis pipelines.

In [13]:
from pathlib import Path

output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

fasta_output.export("pdb_fasta_results", export_path=output_dir, file_format="json")
print(f"Exported to {output_dir / 'pdb_fasta_results.json'}")

Exported to example_output/pdb_fasta_results.json
